# Geopolitical Exposure Proof of Concept

## 01 — Event and data audit

This notebook determines whether the February 24, 2022 pilot can proceed without inspecting event returns. It audits embeddings, identifiers, return-data availability, point-in-time text, and API readiness. A failed gate is a data-acquisition task—not permission to relax the design.


In [1]:
from pathlib import Path
import os,sys,json,hashlib,platform,importlib.util
import pandas as pd,numpy as np
from IPython.display import display
HERE=Path.cwd().resolve()
if HERE.name=='geopolitical_exposure': ROOT=HERE.parent
elif HERE.name=='Asset_Embeddings': ROOT=HERE
else: ROOT=(HERE/'Projects'/'Asset_Embeddings').resolve()
EXT=ROOT/'geopolitical_exposure'; CONFIG=EXT/'config'; SCHEMAS=EXT/'schemas'; MANIFESTS=EXT/'manifests'; RESULTS=EXT/'results'; RAW_DOCS=EXT/'documents'/'raw'
for path in [MANIFESTS,RESULTS,RAW_DOCS]: path.mkdir(parents=True,exist_ok=True)
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as handle:
  for block in iter(lambda:handle.read(1024*1024),b''): h.update(block)
 return h.hexdigest()
with open(MANIFESTS/'research_design_pre_audit.json') as handle: design=json.load(handle)
with open(CONFIG/'russia_ukraine_pilot.json') as handle: event=json.load(handle)
print('Verified pre-audit design:',design['sha256'])


Verified pre-audit design: 9016570954552ba07eb199ce8cab392536e6dd52e515f26b9184b3f0ed284703


## 1. Audit the pre-event embedding representations

The primary snapshot is 2021Q4. We require an all-S12 Binary representation and all four Active Share × turnover groups. The same PERMCO universe must be sufficiently represented across models for a fair comparison.


In [2]:
embedding_paths={
 'all_s12':ROOT/'active_passive_extension/benchmarks/all_s12_control/2021q4/rs_binary/asset_embeddings_10d.parquet',
 'stable_stock_picker':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/stable_stock_picker/rs_binary/asset_embeddings_10d.parquet',
 'tactical_active':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/tactical_active/rs_binary/asset_embeddings_10d.parquet',
 'benchmark_rebalancer':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/benchmark_rebalancer/rs_binary/asset_embeddings_10d.parquet',
 'passive_like':ROOT/'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/passive_like/rs_binary/asset_embeddings_10d.parquet'}
embedding_frames={}; rows=[]
for name,path in embedding_paths.items():
 assert path.exists(),f'Missing {path}'
 frame=pd.read_parquet(path); embedding_frames[name]=frame
 dims=[column for column in frame if column.startswith('embedding_')]
 rows.append({'representation':name,'path':str(path.relative_to(ROOT)),'firms':frame.permco.nunique(),'dimensions':len(dims),'duplicate_permcos':int(frame.permco.duplicated().sum()),'missing_embedding_values':int(frame[dims].isna().sum().sum())})
embedding_audit=pd.DataFrame(rows); common_permcos=set.intersection(*[set(frame.permco) for frame in embedding_frames.values()])
embedding_audit['common_universe_firms']=len(common_permcos); display(embedding_audit)
embedding_audit.to_parquet(RESULTS/'embedding_audit.parquet',index=False)


,representation,path,firms,dimensions,duplicate_permcos,missing_embedding_values,common_universe_firms
0,all_s12,active_passive_extension/benchmarks/all_s12_co...,1805,10,0,0,1805
1,stable_stock_picker,active_passive_extension/benchmarks/behavioral...,1805,10,0,0,1805
2,tactical_active,active_passive_extension/benchmarks/behavioral...,1805,10,0,0,1805
3,benchmark_rebalancer,active_passive_extension/benchmarks/behavioral...,1805,10,0,0,1805
4,passive_like,active_passive_extension/benchmarks/behavioral...,1805,10,0,0,1805


## 2. Audit identifiers and firm controls

Text documents require a separate historical entity-to-PERMCO bridge. The existing CRSP names file provides PERMCO, PERMNO, CUSIP, ticker, and company name, but not SEC CIK. A CIK bridge is therefore a required acquisition item.


In [3]:
names_path=ROOT/'data/crosswalks/crsp/year=2021/quarter=4/stocknames_2021q4.parquet'; controls_path=ROOT/'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet'
names=pd.read_parquet(names_path); controls=pd.read_parquet(controls_path)
common_names=names[names.permco.isin(common_permcos)].sort_values(['permco','nameenddt']).drop_duplicates('permco',keep='last')
identifier_audit=pd.DataFrame([{'embedding_common_firms':len(common_permcos),'firms_with_crsp_name':common_names.permco.nunique(),'name_coverage':common_names.permco.nunique()/len(common_permcos),'firms_with_ticker':common_names.loc[common_names.ticker.notna(),'permco'].nunique(),'firms_with_controls':controls.loc[controls.permco.isin(common_permcos),'permco'].nunique(),'control_coverage':controls.loc[controls.permco.isin(common_permcos),'permco'].nunique()/len(common_permcos),'cik_bridge_available':False}])
display(identifier_audit); identifier_audit.to_parquet(RESULTS/'identifier_audit.parquet',index=False)


,embedding_common_firms,firms_with_crsp_name,name_coverage,firms_with_ticker,firms_with_controls,control_coverage,cik_bridge_available
0,1805,1805,1.0,1805,1803,0.998892,False


## 3. Audit return outcomes without examining the event reaction

Only coverage and dates are examined here. The existing cache contains monthly CRSP returns, which are useful for broad diagnostics but insufficient for the preregistered daily event windows. Daily CRSP returns and factor returns must be downloaded in a later outcome-construction notebook while the seed and model-selection stages remain blind to them.


In [4]:
monthly_path=ROOT/'remaining_benchmarks/data/crsp_monthly_returns/crsp_firm_monthly_2004q1_2025q1.parquet'
monthly=pd.read_parquet(monthly_path,columns=['permco','date']); monthly['date']=pd.to_datetime(monthly.date)
event_months=monthly.loc[monthly.date.between('2021-01-01','2022-12-31') & monthly.permco.isin(common_permcos)]
return_audit=pd.DataFrame([{'monthly_cache_exists':monthly_path.exists(),'monthly_start':monthly.date.min(),'monthly_end':monthly.date.max(),'common_firms_with_2021_2022_monthly_returns':event_months.permco.nunique(),'monthly_coverage':event_months.permco.nunique()/len(common_permcos),'daily_crsp_cache_exists':False,'daily_factor_cache_exists':False,'outcome_values_inspected':False}])
display(return_audit); return_audit.to_parquet(RESULTS/'return_coverage_audit.parquet',index=False)


,monthly_cache_exists,monthly_start,monthly_end,common_firms_with_2021_2022_monthly_returns,monthly_coverage,daily_crsp_cache_exists,daily_factor_cache_exists,outcome_values_inspected
0,True,2004-01-30,2024-12-31,1805,1.0,False,False,False


## 4. Audit the point-in-time text corpus

The adapter accepts SEC filings now and earnings-call transcripts later. The audit counts only files and metadata; it does not score exposure. A usable pilot requires broad pre-event coverage and valid `available_date` fields.


In [5]:
document_files=[path for path in RAW_DOCS.rglob('*') if path.is_file() and path.suffix.lower() in {'.txt','.json','.jsonl','.html','.htm','.pdf'}]
metadata_candidates=[path for path in RAW_DOCS.rglob('*') if path.is_file() and path.suffix.lower() in {'.parquet','.csv','.jsonl'}]
text_audit=pd.DataFrame([{'raw_document_files':len(document_files),'metadata_files':len(metadata_candidates),'pre_event_documents_validated':0,'firms_with_pre_event_text':0,'earnings_call_files':sum('transcript' in path.name.lower() or 'earnings' in path.name.lower() for path in document_files),'sec_filing_files':sum('10-k' in path.name.lower() or '10q' in path.name.lower() or '10-q' in path.name.lower() for path in document_files),'available_date_validated':False}])
display(text_audit); text_audit.to_parquet(RESULTS/'text_corpus_audit.parquet',index=False)


,raw_document_files,metadata_files,pre_event_documents_validated,firms_with_pre_event_text,earnings_call_files,sec_filing_files,available_date_validated
0,0,0,0,0,0,0,False


## 5. API readiness and secure-key check

The API key is never printed, stored, hashed, or written into notebook output. This cell checks only whether `OPENAI_API_KEY` exists in the kernel environment and whether the SDK is installed. API access is not required to complete this audit.


In [6]:
sdk_available=importlib.util.find_spec('openai') is not None
api_audit=pd.DataFrame([{'openai_sdk_available':sdk_available,'openai_api_key_present':bool(os.environ.get('OPENAI_API_KEY')),'model_variable_present':bool(os.environ.get('OPENAI_EXPOSURE_MODEL')),'key_value_logged':False,'api_called':False}])
display(api_audit); api_audit.to_parquet(RESULTS/'api_readiness_audit.parquet',index=False)


,openai_sdk_available,openai_api_key_present,model_variable_present,key_value_logged,api_called
0,False,False,False,False,False


## 6. Point-in-time holdings availability gate

The embeddings exist for 2021Q4, but existence is not sufficient. Before accepting them for a February 24 event, we must document when the underlying S12 reports became public and remove any filing first available after February 23. This gate remains unresolved until a source-date audit is implemented.


In [7]:
holdings_timing=pd.DataFrame([{'snapshot':'2021Q4','embeddings_exist':True,'underlying_report_dates_verified':False,'public_availability_dates_verified':False,'post_cutoff_filers_removed':False,'status':'requires source-date verification'}])
display(holdings_timing); holdings_timing.to_parquet(RESULTS/'holdings_timing_audit.parquet',index=False)


,snapshot,embeddings_exist,underlying_report_dates_verified,public_availability_dates_verified,post_cutoff_filers_removed,status
0,2021Q4,True,False,False,False,requires source-date verification


## 7. Event feasibility gates

The event is accepted only after every required gate passes. API readiness is deliberately not a gate because text extraction can be developed and manually tested without production API calls. No event-return values are loaded during this decision.


In [8]:
gates=pd.DataFrame([
 ('all_embedding_groups',len(embedding_audit)==5 and embedding_audit.firms.min()>=1500,True,'Existing 2021Q4 Binary artifacts'),
 ('common_embedding_universe',len(common_permcos)>=1500,True,'Common PERMCO universe across five representations'),
 ('crsp_identifier_coverage',identifier_audit.name_coverage.iloc[0]>=.95,True,'Existing CRSP names'),
 ('sec_cik_crosswalk',bool(identifier_audit.cik_bridge_available.iloc[0]),True,'Acquire/build historical CIK–PERMCO bridge'),
 ('pre_event_text_coverage',text_audit.firms_with_pre_event_text.iloc[0]>=500,True,'Acquire SEC filings or transcripts'),
 ('available_date_integrity',bool(text_audit.available_date_validated.iloc[0]),True,'Validate public availability dates'),
 ('holdings_public_availability',bool(holdings_timing.public_availability_dates_verified.iloc[0]),True,'Audit S12 filing availability'),
 ('monthly_return_coverage',return_audit.monthly_coverage.iloc[0]>=.95,False,'Existing monthly cache'),
 ('daily_event_returns',bool(return_audit.daily_crsp_cache_exists.iloc[0]),True,'Download later under outcome quarantine'),
 ('daily_factor_returns',bool(return_audit.daily_factor_cache_exists.iloc[0]),True,'Download later under outcome quarantine')],columns=['gate','passed','required','next_action'])
required_pass=bool(gates.loc[gates.required,'passed'].all()); gates['event_accepted']=required_pass
display(gates); print('Candidate event accepted:',required_pass)
gates.to_parquet(RESULTS/'event_feasibility_gates.parquet',index=False)


,gate,passed,required,next_action,event_accepted
0,all_embedding_groups,True,True,Existing 2021Q4 Binary artifacts,False
1,common_embedding_universe,True,True,Common PERMCO universe across five representat...,False
2,crsp_identifier_coverage,True,True,Existing CRSP names,False
3,sec_cik_crosswalk,False,True,Acquire/build historical CIK–PERMCO bridge,False
4,pre_event_text_coverage,False,True,Acquire SEC filings or transcripts,False
5,available_date_integrity,False,True,Validate public availability dates,False
6,holdings_public_availability,False,True,Audit S12 filing availability,False
7,monthly_return_coverage,True,False,Existing monthly cache,False
8,daily_event_returns,False,True,Download later under outcome quarantine,False
9,daily_factor_returns,False,True,Download later under outcome quarantine,False


Candidate event accepted: False


## 8. Freeze the audit result

A failed audit is expected at this stage. It precisely defines the next work package: acquire point-in-time text, build CIK–PERMCO links, verify holdings availability, and prepare quarantined daily outcomes.


In [9]:
audit_payload={'stage':'event_data_audit','research_design_sha256':design['sha256'],'event_id':event['event_id'],'event_accepted':required_pass,'outcome_values_inspected':False,'embedding_representations':embedding_audit.to_dict(orient='records'),'identifier_audit':identifier_audit.astype(object).where(pd.notna(identifier_audit),None).to_dict(orient='records'),'return_coverage':return_audit.astype(object).where(pd.notna(return_audit),None).to_dict(orient='records'),'text_audit':text_audit.to_dict(orient='records'),'holdings_timing':holdings_timing.to_dict(orient='records'),'gates':gates.to_dict(orient='records')}
audit_text=json.dumps(audit_payload,sort_keys=True,default=str); audit_payload['sha256']=hashlib.sha256(audit_text.encode()).hexdigest()
with open(MANIFESTS/'event_data_audit.json','w') as handle: json.dump(audit_payload,handle,indent=2,default=str)
print('Event/data audit SHA-256:',audit_payload['sha256'])
print('Next work package: text acquisition + identifier bridge + holdings availability verification. Do not inspect event returns.')


Event/data audit SHA-256: ff977438cc7c8206a84b014e2f04f38cb2841159dd53473f97b439b2c994b0fd
Next work package: text acquisition + identifier bridge + holdings availability verification. Do not inspect event returns.
